In [1]:
import pandas as pd
import torch
import numpy as np

In [3]:
#Loading in SIMPA datasets -- original and and (syntactic + semantic) simplified

original = pd.read_table('/home/c23068554/final_project/datasets/simpa/ss.original', header=None)
simplified = pd.read_table('/home/c23068554/final_project/datasets/simpa/ss.simplified', header=None)

#Combine datasets into one dataframe
df = pd.DataFrame({'original': original[0], 'simplified': simplified[0]})

#### Test sizes:
3 - 0.0025,
2 - 0.001,
1 - 0.0001

In [4]:
# data splitting, splitting data values to be tested and some to be used as few-shot
from sklearn.model_selection import train_test_split

random_state = 59
test_size = 0.0025

data_test, data_train = train_test_split(df, test_size = test_size, random_state = random_state)

print(len(data_test))
print(len(data_train))

1097
3


# Making prompt with randomized n-shot prompting



In [5]:
instruction = "Please rewrite the following complex sentence in order to make it easier to understand by non-native speakers of English. You can do so by replacing complex words with simpler synonyms (i.e. paraphrasing), deleting unimportant information (i.e. compression), and/or splitting a long complex sentence into several simpler ones. The final simplified sentence needs to be grammatical, fluent, and retain the main ideas of its original counterpart without altering its meaning.\n\n"
def makePrompt(instruction, examples):
  #formatting text for fewshot examples
  fewshot = ""
  for index, row in examples.iterrows():
    fewshot += (f"Complex: {row.loc['original']}\nSimple: {row.loc['simplified']}\n\n")
  return(instruction + fewshot)

fewshot_example = makePrompt(instruction, data_train)
print(fewshot_example)

Please rewrite the following complex sentence in order to make it easier to understand by non-native speakers of English. You can do so by replacing complex words with simpler synonyms (i.e. paraphrasing), deleting unimportant information (i.e. compression), and/or splitting a long complex sentence into several simpler ones. The final simplified sentence needs to be grammatical, fluent, and retain the main ideas of its original counterpart without altering its meaning.

Complex: However, it is advisable to submit your application as soon as practicable to ensure no delay is made should your application be referred to our Licensing Committee.
Simple: It is best to submit your application as soon as possible. This will ensure there is no delay if your application has to be referred to our Licensing Committee.

Complex: If you are not sure who to pay the rent to, you could either carry on paying it to the old landlord or set the money aside in a separate bank account so that you can pay i

# Load in model and inference

In [6]:
# Test for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"compute type: {device}")

compute type: cuda


### Decoder models

In [7]:
# Decoder models
minstral_7b = "mistralai/Mistral-7B-Instruct-v0.3"
llama31_8b = "meta-llama/Meta-Llama-3.1-8B-Instruct"
phi3_14b = "microsoft/Phi-3-medium-4k-instruct"
gemma2_9b = "google/gemma-2-9b-it"
qwen2_7b = "Qwen/Qwen2.5-7B-Instruct"
llama33_70b = "meta-llama/Meta-Llama-3.1-70B-Instruct"

In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(llama31_8b)
model = AutoModelForCausalLM.from_pretrained(llama31_8b).to(device)

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [9]:
def generateDec(prompt):
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, return_dict=False).to(device)

    # Store input length so we can slice it off later
    input_len = input_ids.shape[1]

    output = model.generate(
        input_ids,
        #example on HF sets this as 1000 (may cause leakage)
        max_new_tokens=256,
        pad_token_id=tokenizer.eos_token_id

    )

    # Decoder output includes the input — slice it off
    return tokenizer.decode(
        output[0][input_len:],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    ).strip()

### Encoder-Decoder models

In [10]:
# Encoder-Decoder Models
flanT5small = "google/flan-t5-small"
flanT5large = "google/flan-t5-large"
flanT5xl = "google/flan-t5-xl" #3B parameters
mT5 = "google/mt5-large" #1.2B parameters 

In [11]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

#tokenizer = AutoTokenizer.from_pretrained(flanT5xl)
#model = AutoModelForSeq2SeqLM.from_pretrained(flanT5xl).to(device)

In [12]:
def generateEncDec(prompt):
    input_ids = tokenizer(message = prompt, return_tensors="pt").input_ids.to(device)
    output = model.generate(
        input_ids,
        max_new_tokens = 512,
        #do_sample=False,
        #num_beams=1,
        #encoder_no_repeat_ngram_size=5
    )
    return tokenizer.decode(
        output[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    ).strip()

In [13]:
print(model.generation_config)

GenerationConfig {
  "bos_token_id": 128000,
  "do_sample": true,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "temperature": 0.6,
  "top_p": 0.9
}



In [14]:
#looping through all testing subset
def promptLoop(fewshot_example, data_test):
  LMsimplified = []
  token_counts = []
  for row in data_test['original']:
    # Added note to just simplify 
    full_prompt = fewshot_example + f"Complex: {row}\nSimple:"
    #print(full_prompt)
    token_counts.append(len(tokenizer(full_prompt)['input_ids']))
    LMsimplified.append(generateDec(full_prompt))
    print("done")
  print(token_counts)
  return LMsimplified

In [15]:
print(fewshot_example + f"Complex:\nSimple:")

Please rewrite the following complex sentence in order to make it easier to understand by non-native speakers of English. You can do so by replacing complex words with simpler synonyms (i.e. paraphrasing), deleting unimportant information (i.e. compression), and/or splitting a long complex sentence into several simpler ones. The final simplified sentence needs to be grammatical, fluent, and retain the main ideas of its original counterpart without altering its meaning.

Complex: However, it is advisable to submit your application as soon as practicable to ensure no delay is made should your application be referred to our Licensing Committee.
Simple: It is best to submit your application as soon as possible. This will ensure there is no delay if your application has to be referred to our Licensing Committee.

Complex: If you are not sure who to pay the rent to, you could either carry on paying it to the old landlord or set the money aside in a separate bank account so that you can pay i

### Inference

In [16]:
LMoutput = (promptLoop(fewshot_example, data_test))
#print(promptLoop(fewshot_example, data_test))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done
done


In [17]:
# Clean prompt leakage from decoder outputs
def clean_output(text):
    # Cut off at first "Complex:" which signals prompt repetition
    for delimiter in ['\nComplex:', '\n\nComplex:']:
        if delimiter in text:
            text = text.split(delimiter)[0]
    return text.strip()

LMoutput = [clean_output(s) for s in LMoutput]

### Organising outputs

In [18]:
reference = data_test['simplified'].tolist()
source = data_test['original'].tolist()

In [19]:
import csv

# change to each experiment ID
exp_id = "M-4"
out_path = "lm_output.csv"

with open(out_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(["Exp ID", "Index", "Original", "Reference", "LM Output"])
    for i, (orig, ref, lm) in enumerate(zip(source, reference, LMoutput)):
        writer.writerow([exp_id, i, orig, ref, lm])

print(f"Wrote {len(source)} rows to {out_path}")

Wrote 1097 rows to lm_output.csv


# Evaluation

### Preliminary tests on the model
 How many sentences dont actually get simplified by the model?

In [20]:
count = 0
for original, simple, lmSimple in zip(data_test['original'], data_test['simplified'], LMoutput):
  print(original)
  print(simple)
  print(lmSimple + "\n")
  if original == lmSimple:
    #print(original)
    #print(simple)
    #print(lmSimple)
    count += 1
print(f"Number of sentences not simplified or altered by the model: {count}")


The exhibitions served a number of purposes - their main focus was to promote business and industry, open up new markets, and generally to outperform competitors, in an increasingly global economic market.
The exhibitions served a number of purposes. The exhibitions main focus was to promote business and industry, open up new markets, and generally to outperform competitors, in an increasingly global economic market
The exhibitions aimed to promote business, open new markets, and outperform competitors in a global economy.

To aid biodiversity conservation we have drawn up Habitat Action Plans (HAPs) for grassland, woodland, heathland and wetland habitats across 130 target sites in Sheffield.
To aid biodiversity conservation we have drawn up Habitat Action Plans (HAPs) for grassland, woodland, heathland and wetland habitats across 130 target sites in Sheffield.
We have created plans to protect different habitats across 130 sites in Sheffield.

If notice of the sale hasn ’ t been given,

### ROUGE, BLEU, BERTScore, SARI

In [21]:
import evaluate

#load metrics
rouge = evaluate.load('rouge')
bleu = evaluate.load('bleu')
bertscore = evaluate.load('bertscore')
sari = evaluate.load('sari')

#Converting simplified reference sentences into a list inside a list
sari_references = [[s] for s in data_test["simplified"].astype(str).tolist()]
sari_score = sari.compute(sources= source, predictions= LMoutput, references= sari_references)

# Compute scores
rouge_results = rouge.compute(predictions= LMoutput, references= reference)
bleu_results = bleu.compute(predictions= LMoutput, references= reference)

bertscore_compute = bertscore.compute(predictions=LMoutput, references= reference, lang='en')
berstcoreAvg = np.mean(bertscore_compute['f1'])

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Flesch-Kincaid Score

In [22]:
import textstat
textstat.set_lang("en")

originalAvg   = np.mean([textstat.flesch_reading_ease(s) for s in data_test["original"].tolist()])
simplifiedAvg = np.mean([textstat.flesch_reading_ease(s) for s in LMoutput])

### LENS score
<p>Using script with different environment due to package conflicts</p>

In [24]:
import subprocess

lens_venv = "/home/c23068554/miniconda3/envs/lens_eval/bin/python"
result = subprocess.run([lens_venv, "lens_score.py", "lm_output.csv"], capture_output= True, text = True)
lensAvg = float(result.stdout.strip().split('\n')[-1])

In [25]:
#output
print(f"Dataset: SIMPA, size: {len(data_test)}, random state: {random_state}")
print()
print(f"ROUGE Score: {rouge_results['rouge1']}")
print(f"BLEU Score: {bleu_results['bleu']}")
print(f"BERTScore Score: {berstcoreAvg}")
print(f"Sari Score: {sari_score['sari']}")
print()
print(f"Original Flesch score: {originalAvg}")
print(f"Simplified Flesch score: {simplifiedAvg}")
print(f"Average LENS score: {lensAvg}")
print()
print(f"unsimplified sentences: {count}/{len(data_test)}")

Dataset: SIMPA, size: 1097, random state: 59
Model:google/flan-t5-large
Prompt: BLESS 2

ROUGE Score: 0.5706375078916119
BLEU Score: 0.22046508640868137
BERTScore Score: 0.9365719342188283
Sari Score: 35.50608225586582

Original Flesch score: 38.093855475465475
Simplified Flesch score: 50.30276664155593
Average LENS score: 65.72250815669464

unsimplified sentences: 1/1097


In [26]:
print(f"\nTriple click and paste into first metric column")
print(f"{sari_score['sari']:.4f}\t{rouge_results['rouge1']:.4f}\t{bleu_results['bleu']:.4f}\t{berstcoreAvg:.4f}\t{lensAvg:.4f}\t{simplifiedAvg:.4f}\t{count}/{len(data_test)}")


Triple click and paste into first metric column
35.5061	0.5706	0.2205	0.9366	65.7225	50.3028	1/1097


In [27]:
# command to clear cache often, to reduce disk space used:
# rm -rf ~/.cache/*

#check disk space used:
# du -h --max-depth=1 ~ | sort -h

'''
Exporting CSV:
On local:
scp -r c23068554@10.98.84.2:/home/c23068554/final_project/lm_output.csv '/Users/justinwoodham/Desktop/CS/Y3/Final Year Project/raw outputs'
On Remote: 
rm lm_output.csv

'''

"\nExporting CSV:\nOn local:\nscp -r c23068554@10.98.84.2:/home/c23068554/final_project/lm_output.csv '/Users/justinwoodham/Desktop/CS/Y3/Final Year Project/raw outputs'\nOn Remote: \nrm lm_output.csv\n\n"